# Treino do MARCOS

Ensina o Gemma 4 E4B a responder no seu jeito, usando os exemplos de `C:\Marcos-dados\finetune`.

**Antes de tudo:** menu *Ambiente de execução > Alterar o tipo de ambiente* e escolha **GPU T4**.

Rode as células em ordem. Se a sessão cair, rode tudo de novo: o treino continua do último ponto salvo no seu Drive.

## 1. Instalar

In [ ]:
!pip install -q unsloth

## 2. Pastas

Só os pontos de salvamento e o modelo final vão para o Drive. Suas conversas ficam na sessão e somem quando ela fecha.

In [ ]:
import os
from google.colab import drive

drive.mount("/content/drive")
PASTA_DRIVE = "/content/drive/MyDrive/marcos-treino"
PASTA_CHECKPOINTS = f"{PASTA_DRIVE}/checkpoints"
os.makedirs(PASTA_CHECKPOINTS, exist_ok=True)

## 3. Enviar os exemplos

Escolha os dois arquivos de `C:\Marcos-dados\finetune`: `treino.jsonl` e `validacao.jsonl`.

In [ ]:
from google.colab import files

faltando = [n for n in ("treino.jsonl", "validacao.jsonl") if not os.path.exists(n)]
if faltando:
    files.upload()
assert os.path.exists("treino.jsonl") and os.path.exists("validacao.jsonl"), "Envie os dois arquivos"

## 4. Configurações

In [ ]:
QUANTIDADE_TREINO = 6000     # exemplos usados; mais exemplos = treino mais longo
QUANTIDADE_VALIDACAO = 300
MAX_TOKENS = 1024
LOTE = 2
ACUMULAR = 4                 # o lote efetivo é LOTE x ACUMULAR = 8 exemplos por passo
PASSOS = QUANTIDADE_TREINO // (LOTE * ACUMULAR)
print(f"{PASSOS} passos de treino")

## 5. Carregar o modelo

In [ ]:
from unsloth import FastModel

model, tokenizer = FastModel.from_pretrained(
    model_name="unsloth/gemma-4-E4B-it",
    max_seq_length=MAX_TOKENS,
    load_in_4bit=True,       # QLoRA: cabe nos 16 GB da T4
    full_finetuning=False,
)

model = FastModel.get_peft_model(
    model,
    finetune_vision_layers=False,   # só texto
    finetune_language_layers=True,
    finetune_attention_modules=True,
    finetune_mlp_modules=True,
    r=16,
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    random_state=3407,
)

## 6. Preparar os exemplos

In [ ]:
from datasets import load_dataset
from unsloth.chat_templates import get_chat_template

tokenizer = get_chat_template(tokenizer, chat_template="gemma-4")

dados = load_dataset("json", data_files={"treino": "treino.jsonl", "validacao": "validacao.jsonl"})
treino = dados["treino"].shuffle(seed=42).select(range(min(QUANTIDADE_TREINO, len(dados["treino"]))))
validacao = dados["validacao"].shuffle(seed=42).select(range(min(QUANTIDADE_VALIDACAO, len(dados["validacao"]))))


def formatar(lote):
    textos = [
        tokenizer.apply_chat_template(mensagens, tokenize=False, add_generation_prompt=False).removeprefix("<bos>")
        for mensagens in lote["messages"]
    ]
    return {"text": textos}


treino = treino.map(formatar, batched=True, remove_columns=treino.column_names)
validacao = validacao.map(formatar, batched=True, remove_columns=validacao.column_names)
print(treino[0]["text"][:600])

## 7. Treinar

O modelo só aprende com as **suas** respostas: as falas dos outros entram apenas como contexto.

Uma *loss* entre 13 e 15 no começo é normal para o E4B. O importante é ela cair.

In [ ]:
import glob

from trl import SFTConfig, SFTTrainer
from unsloth.chat_templates import train_on_responses_only

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=treino,
    eval_dataset=validacao,
    args=SFTConfig(
        dataset_text_field="text",
        max_length=MAX_TOKENS,
        per_device_train_batch_size=LOTE,
        gradient_accumulation_steps=ACUMULAR,
        max_steps=PASSOS,
        warmup_steps=20,
        learning_rate=2e-4,
        lr_scheduler_type="linear",
        optim="adamw_8bit",
        weight_decay=0.001,
        logging_steps=10,
        eval_strategy="steps",
        eval_steps=250,
        save_strategy="steps",
        save_steps=100,              # salva no Drive a cada 100 passos
        save_total_limit=2,
        output_dir=PASTA_CHECKPOINTS,
        seed=3407,
        report_to="none",
    ),
)
trainer = train_on_responses_only(
    trainer,
    instruction_part="<|turn>user\n",
    response_part="<|turn>model\n",
)

continuar = bool(glob.glob(f"{PASTA_CHECKPOINTS}/checkpoint-*"))
print("Continuando do último ponto salvo" if continuar else "Começando do zero")
trainer.train(resume_from_checkpoint=continuar)

## 8. Testar

In [ ]:
from transformers import TextStreamer

SISTEMA = "Você é o Markin conversando no WhatsApp. Responda do jeito que ele responderia."


def conversar(texto):
    mensagens = [{"role": "system", "content": SISTEMA}, {"role": "user", "content": texto}]
    entrada = tokenizer.apply_chat_template(
        mensagens, add_generation_prompt=True, tokenize=True, return_dict=True, return_tensors="pt"
    ).to("cuda")
    model.generate(
        **entrada, max_new_tokens=80, use_cache=True,
        temperature=1.0, top_p=0.95, top_k=64,
        streamer=TextStreamer(tokenizer, skip_prompt=True),
    )


for pergunta in ["e ai mano, bora jogar hj?", "q moto tu vai comprar?", "to triste hoje"]:
    print(f"> {pergunta}")
    conversar(pergunta)
    print()

## 9. Exportar para o Ollama

Gera o arquivo `.gguf` (cerca de 5 GB) e copia para `marcos-treino/modelo` no seu Drive. Demora uns 20 minutos.

In [ ]:
import shutil

model.save_pretrained(f"{PASTA_DRIVE}/lora")     # o ajuste em si, pequeno, para treinar mais depois
tokenizer.save_pretrained(f"{PASTA_DRIVE}/lora")

model.save_pretrained_gguf("modelo", tokenizer, quantization_method="q4_k_m")
os.makedirs(f"{PASTA_DRIVE}/modelo", exist_ok=True)
for arquivo in glob.glob("modelo/*.gguf") + glob.glob("modelo/Modelfile"):
    shutil.copy(arquivo, f"{PASTA_DRIVE}/modelo/")
print(os.listdir(f"{PASTA_DRIVE}/modelo"))

## 10. Apagar suas conversas da sessão

In [ ]:
for arquivo in ("treino.jsonl", "validacao.jsonl"):
    if os.path.exists(arquivo):
        os.remove(arquivo)
print("Exemplos apagados desta sessão.")